In [24]:
pip install openml

Note: you may need to restart the kernel to use updated packages.


In [25]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [44]:
import openml
import pandas as pd
from tqdm import tqdm

In [45]:
datasets = openml.datasets.list_datasets(output_format='dataframe')

In [46]:
datasets = datasets[0:1]

In [47]:
datasets.shape

(1, 16)

In [48]:
# datasets = datasets[['did', 'name', 'NumberOfInstances', 'NumberOfFeatures', 'NumberOfClasses']]

In [187]:
results = []

In [188]:
task_type_filter = 'Supervised Classification'

In [189]:
# task_ids_to_process = [
#     279, 267, 273, 3053, 3054, 3055, 254, 260, 245, 271, 2119, 2125, 232, 288, 2120, 2121, 241, 258, 253, 3044, 336, 2122, 256, 262, 
#     236, 2356, 340, 75219, 75098, 189864, 189863, 189869, 190155, 146583
# ]
task_ids_to_process = [
    340, 75219, 75098, 189864, 189863, 189869, 190155, 146583
]

In [16]:
for dataset_id in tqdm(datasets['did'], desc='Processing Datasets', position=0):
    try:
        # Get tasks associated with the dataset
        tasks = openml.tasks.list_tasks(data_id=dataset_id, output_format='dataframe')
        
        # Filter tasks pick only the first task per dataset
        classification_tasks = tasks[tasks['task_type'] == task_type_filter]
        print(len(classification_tasks))

        if len(classification_tasks) > 0:
            task_id = classification_tasks.iloc[0]['tid']  
            task_type = classification_tasks.iloc[0]['task_type'] 
            print(f"taskid: {task_id}")

            # Get model runs
            runs = openml.runs.list_runs(task=[task_id], output_format='dataframe')
            print(f"runs: {len(runs)}")

            # Initialize a list to store accuracy for each run
            accuracy_list = []
            for run_info in tqdm(runs.itertuples(index=False), total=len(runs), desc='Processing Runs', position=1, leave=False):
                run_id = run_info.run_id
                
                # Get the performance of the run
                run_results = openml.runs.get_run(run_id)
                accuracy = run_results.evaluations.get('predictive_accuracy', None)  # Some tasks might not have accuracy
                
                accuracy_list.append({
                    'run_id': run_id,
                    'accuracy': accuracy,
                    'flow_id': run_info.flow_id
                })
            
            # Convert to DataFrame
            accuracy_df = pd.DataFrame(accuracy_list)

            # Remove duplicate entries
            accuracy_df = accuracy_df.drop_duplicates(subset=['flow_id', 'accuracy'])

            # Sort by accuracy in descending order and select the top N runs
            # top_runs = accuracy_df.sort_values(by='accuracy', ascending=False).head(max_runs)

            # Append details to results
            for _, run in tqdm(accuracy_df.iterrows(), desc='Appending top results', leave=False):
                flow = openml.flows.get_flow(run['flow_id'])  # Fetch the flow for model details
                results.append({
                    'dataset_id': dataset_id,
                    'dataset_name': datasets[datasets['did'] == dataset_id]['name'].values[0],
                    'num_instances': datasets[datasets['did'] == dataset_id]['NumberOfInstances'].values[0],
                    'num_features': datasets[datasets['did'] == dataset_id]['NumberOfFeatures'].values[0],
                    'num_classes': datasets[datasets['did'] == dataset_id]['NumberOfClasses'].values[0],
                    'model_id': flow.name,
                    'accuracy': run['accuracy'],
                    'task_type': task_type  # Include task type in results
                })
    except Exception as e:
        tqdm.write(f"Error processing dataset {dataset_id}: {e}")  # Using tqdm.write to avoid messing up the progress bar
        continue


Processing Datasets:   0%|                                | 0/1 [00:00<?, ?it/s]

10
taskid: 2
runs: 5520



Processing Datasets:   0%|                                | 0/1 [00:11<?, ?it/s]


KeyboardInterrupt: 

In [190]:
for task_id in tqdm(task_ids_to_process, desc='Processing Tasks', position=0):
    try:
        # Get tasks 
        task = openml.tasks.get_task(task_id)

        # Get model runs
        runs = openml.runs.list_runs(task=[task_id], output_format='dataframe')
        print(f"runs: {len(runs)}")
        
        # Initialize a list to store accuracy for each run
        accuracy_list = []
        for run_info in tqdm(runs.itertuples(index=False), total=len(runs), desc='Processing Runs', position=1, leave=False):
            run_id = run_info.run_id
                
                # Get the performance of the run
            run_results = openml.runs.get_run(run_id)
            accuracy = run_results.evaluations.get('predictive_accuracy', None)  # Some tasks might not have accuracy
                
            accuracy_list.append({
                'run_id': run_id,
                'accuracy': accuracy,
                'flow_id': run_info.flow_id
            })
            
        # Convert to DataFrame
        accuracy_df = pd.DataFrame(accuracy_list)

        # Remove duplicate entries
        accuracy_df = accuracy_df.drop_duplicates(subset=['flow_id', 'accuracy'])

        # Get the dataset associated with the task
        dataset_id = task.dataset_id
        dataset = openml.datasets.get_dataset(dataset_id)

        # Append details to results
        for _, run in tqdm(accuracy_df.iterrows(), total=accuracy_df.shape[0], desc='Appending results', leave=False):
            flow = openml.flows.get_flow(run['flow_id'])  # Fetch the flow for model details
            results.append({
            'dataset_id': dataset_id,
            'dataset_name': dataset.name,
            'num_instances': dataset.qualities['NumberOfInstances'],
            'num_features': dataset.qualities['NumberOfFeatures'],
            'num_classes': dataset.qualities['NumberOfClasses'],
            'default_target_attribute': dataset.default_target_attribute,
            'total_missing_values': dataset.qualities['NumberOfMissingValues'],
            'percentage_missing_values': dataset.qualities['PercentageOfMissingValues'],
            'instances_with_missing_values': dataset.qualities['NumberOfInstancesWithMissingValues'],
            'num_numeric_features': dataset.qualities['NumberOfNumericFeatures'],
            'num_symbolic_features': dataset.qualities['NumberOfSymbolicFeatures'],
            'percentage_numeric_features': dataset.qualities['PercentageOfNumericFeatures'],
            'percentage_symbolic_features': dataset.qualities['PercentageOfSymbolicFeatures'],
            'majority_class_percentage': dataset.qualities['MajorityClassPercentage'],
            'minority_class_percentage': dataset.qualities['MinorityClassPercentage'],
            'majority_class_size': dataset.qualities['MajorityClassSize'],
            'minority_class_size': dataset.qualities['MinorityClassSize'],
            'mean': dataset.qualities['MeanMeansOfNumericAtts'],
            'std_dev': dataset.qualities['MeanStdDevOfNumericAtts'],
            'min': dataset.qualities['MinMeansOfNumericAtts'],
            'max': dataset.qualities['MaxMeansOfNumericAtts'],
            'kurtosis': dataset.qualities['MeanKurtosisOfNumericAtts'],
            'skewness': dataset.qualities['MeanSkewnessOfNumericAtts'],
            # 'evaluation_metrics': {
            #     'naive_bayes_auc': dataset.qualities['NaiveBayesAUC'],
            #     'decision_stump_auc': dataset.qualities['DecisionStumpAUC'],
            #     'j48_auc': dataset.qualities['J48.00001.AUC'],
            #     'kNN_auc': dataset.qualities['kNN1NAUC']
            #     # Add more metrics as needed
            # },
            'auto_correlation': dataset.qualities['AutoCorrelation'],
            'class_entropy': dataset.qualities['ClassEntropy'], 
            'model_id': flow.id,
            'model_name': flow.name,
            'accuracy': run['accuracy'],
            'task_type': task.task_type  # Include task type in results
            })
    except Exception as e:
        tqdm.write(f"Error processing dataset {dataset_id}: {e}")  # Using tqdm.write to avoid messing up the progress bar
        continue


Processing Tasks:   0%|                                   | 0/8 [00:00<?, ?it/s]

runs: 4



Processing Runs: 100%|████████████████████████████| 4/4 [00:02<00:00,  1.57it/s]
                                                                                
Processing Tasks:  12%|███▍                       | 1/8 [00:05<00:37,  5.36s/it]

runs: 3



Processing Runs: 100%|████████████████████████████| 3/3 [00:02<00:00,  1.43it/s]
                                                                                
Processing Tasks:  25%|██████▊                    | 2/8 [00:11<00:33,  5.53s/it]

...


Processing Runs: 100%|████████████████████████████| 1/1 [00:00<00:00,  1.50it/s]
                                                                                
Processing Tasks:  75%|████████████████████▎      | 6/8 [00:29<00:09,  4.64s/it]

runs: 1



Processing Runs: 100%|████████████████████████████| 1/1 [00:00<00:00,  1.62it/s]
                                                                                
Processing Tasks:  88%|███████████████████████▋   | 7/8 [00:33<00:04,  4.31s/it]

runs: 1



Processing Runs: 100%|████████████████████████████| 1/1 [00:00<00:00,  1.50it/s]
                                                                                
Processing Tasks: 100%|███████████████████████████| 8/8 [00:37<00:00,  4.68s/it]


In [191]:
results_df = pd.DataFrame(results)
results_df

,dataset_id,dataset_name,num_instances,num_features,num_classes,default_target_attribute,total_missing_values,percentage_missing_values,instances_with_missing_values,num_numeric_features,...,min,max,kurtosis,skewness,auto_correlation,class_entropy,model_id,model_name,accuracy,task_type
0,155,pokerhand,829201.0,11.0,10.0,class,0.0,0.000000,0.0,5.0,...,2.613158,11.385761,0.246195,0.002590,0.745363,1.416953,365,weka.J48,0.997774,Supervised Classification
1,155,pokerhand,829201.0,11.0,10.0,class,0.0,0.000000,0.0,5.0,...,2.613158,11.385761,0.246195,0.002590,0.745363,1.416953,523,weka.Bagging_J48,0.997745,Supervised Classification
2,155,pokerhand,829201.0,11.0,10.0,class,0.0,0.000000,0.0,5.0,...,2.613158,11.385761,0.246195,0.002590,0.745363,1.416953,523,weka.Bagging_J48,0.997749,Supervised Classification
3,1471,eeg-eye-state,14980.0,15.0,2.0,Class,0.0,0.000000,0.0,14.0,...,4009.767694,4644.022379,8904.721902,71.737266,0.998465,0.992422,18748,sklearn.pipeline.Pipeline(imputer=sklearn.impu...,0.821768,Supervised Classification
4,1471,eeg-eye-state,14980.0,15.0,2.0,Class,0.0,0.000000,0.0,14.0,...,4009.767694,4644.022379,8904.721902,71.737266,0.998465,0.992422,18748,sklearn.pipeline.Pipeline(imputer=sklearn.impu...,0.825410,Supervised Classification
5,1471,eeg-eye-state,14980.0,15.0,2.0,Class,0.0,0.000000,0.0,14.0,...,4009.767694,4644.022379,8904.721902,71.737266,0.998465,0.992422,18748,sklearn.pipeline.Pipeline(imputer=sklearn.impu...,0.826421,Supervised Classification
6,554,mnist_784,70000.0,785.0,10.0,class,0.0,0.000000,0.0,784.0,...,0.000000,140.015057,4894.714443,31.079436,0.095644,3.319837,9725,keras.engine.sequential.Sequential.A9B07AA00BF...,0.979481,Supervised Classification
7,554,mnist_784,70000.0,785.0,10.0,class,0.0,0.000000,0.0,784.0,...,0.000000,140.015057,4894.714443,31.079436,0.095644,3.319837,10571,torch.nn.modules.container.Sequential.8c3a9515...,0.808312,Supervised Classification
8,41145,philippine,5832.0,309.0,2.0,class,0.0,0.000000,0.0,308.0,...,-1.142981,202871.809156,102.364763,2.723289,0.499571,1.000000,17315,sklearn.pipeline.Pipeline(columntransformer=sk...,0.711538,Supervised Classification
9,41144,madeline,3140.0,260.0,2.0,class,0.0,0.000000,0.0,259.0,...,475.897134,517.308917,0.062025,0.034018,0.497929,0.999976,17315,sklearn.pipeline.Pipeline(columntransformer=sk...,0.623552,Supervised Classification


In [192]:
# Append DataFrame to the existing CSV file
csv_file_path = 'results.csv'  # Specify your existing file path here
results_df.to_csv(csv_file_path, mode='a', index=False, header=False)  # Ensure header=False to append correctly

In [29]:
import openml

# Replace with your task ID
task_id = 267 # e.g., 1234

# Fetch runs associated with the task
runs = openml.runs.list_runs(task=[task_id], output_format='dataframe')

# Print the length of runs
print(f"Number of runs associated with task ID {task_id}: {len(runs)}")


Number of runs associated with task ID 267: 372


In [56]:
import openml

def get_task_and_dataset_info(task_id):
    # Get task information
    task = openml.tasks.get_task(task_id)
    
    # Extract relevant task details
    task_info = {
        'task_id': task_id,
        'task_type': task.task_type,  # E.g., Supervised Classification
        'target_name': task.target_name,  # Target variable for classification
        'evaluation_metric': task.evaluation_measure,  # Metric used for evaluation (e.g., accuracy)
        'task_url': task.openml_url,  # URL to the task on OpenML
    }
    
    # Get the dataset associated with the task
    dataset_id = task.dataset_id
    dataset = openml.datasets.get_dataset(dataset_id)
    print("\nDataset :")
    for key, value in dataset.qualities.items():
        print(f"{key}: {value}")

    # print(f"dataset:{dataset.qualities}")
    
    # Extract relevant dataset details
    dataset_info = {
        'dataset_id': dataset_id,
        'dataset_name': dataset.name,
        'num_instances': dataset.qualities['NumberOfInstances'],  # Total number of instances (rows)
        'num_features': dataset.qualities['NumberOfFeatures'],  # Number of features (columns)
        'num_classes': dataset.qualities.get('NumberOfClasses', 'N/A'),  # Number of classes (for classification tasks)
        'dataset_url': dataset.openml_url,  # URL to the dataset on OpenML
        'missing_values': dataset.qualities.get('NumberOfMissingValues', 0),  # Count of missing values
        'default_target_attribute': dataset.default_target_attribute,  # Target attribute for classification
    }
    
    # Combine both task and dataset information
    combined_info = {
        'task_info': task_info,
        'dataset_info': dataset_info
    }
    
    return combined_info

# Example: Get information for a specific task ID
task_id = 267  # Replace with your desired task ID
task_dataset_info = get_task_and_dataset_info(task_id)

# Output task and dataset information
print("Task Information:")
for key, value in task_dataset_info['task_info'].items():
    print(f"{key}: {value}")

print("\nDataset Information:")
for key, value in task_dataset_info['dataset_info'].items():
    print(f"{key}: {value}")



Dataset :
AutoCorrelation: 0.5501955671447197
CfsSubsetEval_DecisionStumpAUC: 0.7186641791044776
CfsSubsetEval_DecisionStumpErrRate: 0.26953125
CfsSubsetEval_DecisionStumpKappa: 0.38164732240097077
CfsSubsetEval_NaiveBayesAUC: 0.7186641791044776
CfsSubsetEval_NaiveBayesErrRate: 0.26953125
CfsSubsetEval_NaiveBayesKappa: 0.38164732240097077
CfsSubsetEval_kNN1NAUC: 0.7186641791044776
CfsSubsetEval_kNN1NErrRate: 0.26953125
CfsSubsetEval_kNN1NKappa: 0.38164732240097077
ClassEntropy: 0.9331343166407831
DecisionStumpAUC: 0.7111194029850746
DecisionStumpErrRate: 0.2721354166666667
DecisionStumpKappa: 0.39105890922334524
Dimensionality: 0.01171875
EquivalentNumberOfAtts: nan
J48.00001.AUC: 0.7074365671641791
J48.00001.ErrRate: 0.2760416666666667
J48.00001.Kappa: 0.38177399756986635
J48.0001.AUC: 0.7074365671641791
J48.0001.ErrRate: 0.2760416666666667
J48.0001.Kappa: 0.38177399756986635
J48.001.AUC: 0.7074365671641791
J48.001.ErrRate: 0.2760416666666667
J48.001.Kappa: 0.38177399756986635
Majori

In [205]:
df = pd.read_csv('./Classification_dataset - Sheet1.csv')
df

,dataset_id,dataset_name,num_instances,num_features,num_classes,default_target_attribute,total_missing_values,percentage_missing_values,instances_with_missing_values,num_numeric_features,...,min,max,kurtosis,skewness,auto_correlation,class_entropy,model_id,model_name,accuracy,task_type
0,50,tic-tac-toe,958,10,2,Class,0,0.000000,0,0,...,NaN,NaN,NaN,NaN,0.998955,0.930954,1068,weka.J48,0.829114,Supervised Classification
1,50,tic-tac-toe,958,10,2,Class,0,0.000000,0,0,...,NaN,NaN,NaN,NaN,0.998955,0.930954,1069,weka.ZeroR,0.617089,Supervised Classification
2,50,tic-tac-toe,958,10,2,Class,0,0.000000,0,0,...,NaN,NaN,NaN,NaN,0.998955,0.930954,1070,weka.Ridor,0.908228,Supervised Classification
3,50,tic-tac-toe,958,10,2,Class,0,0.000000,0,0,...,NaN,NaN,NaN,NaN,0.998955,0.930954,1071,weka.OneR,0.724684,Supervised Classification
4,50,tic-tac-toe,958,10,2,Class,0,0.000000,0,0,...,NaN,NaN,NaN,NaN,0.998955,0.930954,1072,weka.Prism,0.987342,Supervised Classification
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3474,41145,philippine,5832,309,2,class,0,0.000000,0,308,...,-1.142981,202871.809200,102.364763,2.723289,0.499571,1.000000,17315,sklearn.pipeline.Pipeline(columntransformer=sk...,0.711538,Supervised Classification
3475,41144,madeline,3140,260,2,class,0,0.000000,0,259,...,475.897134,517.308917,0.062025,0.034018,0.497929,0.999976,17315,sklearn.pipeline.Pipeline(columntransformer=sk...,0.623552,Supervised Classification
3476,41160,rl,31406,23,2,class,29756,4.119401,17204,8,...,6.995256,1978.971001,6.252849,1.699151,0.999395,0.454125,17322,sklearn.pipeline.Pipeline(columntransformer=sk...,0.908810,Supervised Classification
3477,41990,GTSRB-HueHist,51839,257,43,class,0,0.000000,0,256,...,0.000010,0.075343,97.799811,5.792628,0.765288,5.020278,17315,sklearn.pipeline.Pipeline(columntransformer=sk...,0.085935,Supervised Classification
